In [3]:
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader


In [5]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Lambda(lambda x: x.view(-1)) ])

In [7]:

train_dataset = datasets.FashionMNIST(
    root="./data",
    train=True,
    download=True,
    transform=transform)

100.0%
100.0%
100.0%
100.0%


In [8]:
test_dataset = datasets.FashionMNIST(
    root="./data",
    train=False,
    download=True,
    transform=transform
)


In [9]:
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64)

In [10]:
def softmax(z):
    z = z - torch.max(z, dim=1, keepdim=True)[0]  # numerical stability
    exp_z = torch.exp(z)
    return exp_z / torch.sum(exp_z, dim=1, keepdim=True)

In [11]:
def cross_entropy(predictions, targets):
    m = targets.shape[0]
    log_likelihood = -torch.log(predictions[range(m), targets])
    loss = torch.sum(log_likelihood) / m
    return loss

In [12]:
def forward(X, W, b):
    logits = X @ W.T + b
    probs = softmax(logits)
    return probs

In [13]:
def compute_gradients(X, y, probs, W):
    m = X.shape[0]

    y_onehot = torch.zeros_like(probs)
    y_onehot[range(m), y] = 1

    dz = (probs - y_onehot) / m

    dW = dz.T @ X
    db = torch.sum(dz, dim=0)

    return dW, db

In [14]:
def update_params(W, b, dW, db, lr):
    W -= lr * dW
    b -= lr * db
    return W, b

In [15]:
num_features = 784
num_classes = 10

W = torch.randn(num_classes, num_features) * 0.01
b = torch.zeros(num_classes)

learning_rate = 0.1
epochs = 10

In [16]:
for epoch in range(epochs):

    total_loss = 0

    for X, y in train_loader:

        probs = forward(X, W, b)

        loss = cross_entropy(probs, y)

        dW, db = compute_gradients(X, y, probs, W)

        W, b = update_params(W, b, dW, db, learning_rate)

        total_loss += loss.item()

    print(f"Epoch {epoch+1}, Loss: {total_loss/len(train_loader):.4f}")

Epoch 1, Loss: 0.6231
Epoch 2, Loss: 0.4917
Epoch 3, Loss: 0.4633
Epoch 4, Loss: 0.4490
Epoch 5, Loss: 0.4415
Epoch 6, Loss: 0.4356
Epoch 7, Loss: 0.4286
Epoch 8, Loss: 0.4231
Epoch 9, Loss: 0.4220
Epoch 10, Loss: 0.4186


In [17]:
correct = 0
total = 0

for X, y in test_loader:

    probs = forward(X, W, b)

    predictions = torch.argmax(probs, dim=1)

    correct += (predictions == y).sum().item()
    total += y.size(0)

accuracy = correct / total

print("Test Accuracy:", accuracy)

Test Accuracy: 0.8383
